In [1]:
# Explanation: This cell generates all homomorphic images of the 4-uniform tight cycle C_11 on at most 6 vertices, then excludes those images from the theory.

from collections import defaultdict
from itertools import permutations

target_size = 6
cycle_length = 11
uniformity = 4


def tight_cycle_edges(n, r=4):
    return [tuple((i + j) % n for j in range(r)) for i in range(n)]


def canonical_edge_set(num_vertices, edges):
    """Return a canonical representative of an unlabeled 4-graph."""
    best = None
    for perm in permutations(range(num_vertices)):
        relabeled = tuple(sorted({tuple(sorted(perm[v] for v in edge)) for edge in edges}))
        if best is None or relabeled < best:
            best = relabeled
    return best


def generate_tight_cycle_homomorphic_images(n, max_vertices, r=4):
    """Generate all homomorphic images of C_n^(r) with at most max_vertices vertices, up to isomorphism."""
    cycle_edges = tight_cycle_edges(n, r)
    images = defaultdict(dict)
    assignment = [None] * n
    assignment[0] = 0

    def complete_edge_is_valid(edge):
        colors = [assignment[v] for v in edge]
        if any(color is None for color in colors):
            return True
        return len(set(colors)) == r

    def partial_assignment_is_valid():
        return all(complete_edge_is_valid(edge) for edge in cycle_edges)

    def record_image(current_max):
        num_vertices = current_max + 1
        edges = tuple(sorted({tuple(sorted(assignment[v] for v in edge)) for edge in cycle_edges}))
        key = canonical_edge_set(num_vertices, edges)
        images[num_vertices].setdefault(key, [list(edge) for edge in key])

    def backtrack(pos, current_max):
        if pos == n:
            if partial_assignment_is_valid():
                record_image(current_max)
            return

        upper = min(current_max + 1, max_vertices - 1)
        for color in range(upper + 1):
            assignment[pos] = color
            if partial_assignment_is_valid():
                backtrack(pos + 1, max(current_max, color))
            assignment[pos] = None

    backtrack(1, 0)
    return images


C11_images_by_size = generate_tight_cycle_homomorphic_images(cycle_length, target_size, uniformity)
C11_homomorphic_images = [
    (num_vertices, edges)
    for num_vertices in sorted(C11_images_by_size)
    for edges in sorted(C11_images_by_size[num_vertices].values())
]
C11_image_theories = [
    FourGraphTheory.p(num_vertices, edges=edges)
    for num_vertices, edges in C11_homomorphic_images
]

edge = FourGraphTheory(4, edges=[[0, 1, 2, 3]])
FourGraphTheory.exclude(C11_image_theories)
FGT = FourGraphTheory

print("Homomorphic images of 4-uniform C_11 with at most 6 vertices:")
for num_vertices in range(1, target_size + 1):
    print(f"and size {num_vertices}: ", len(C11_images_by_size.get(num_vertices, {})))
print("total: ", len(C11_homomorphic_images))

print("Number of structures without these C_11 homomorphic images")
print("and size 5: ", len(FGT.generate(5)))
print("and size 6: ", len(FGT.generate(6)))
# print("and size 7: ", len(FGT.generate(7)))


Homomorphic images of 4-uniform C_11 with at most 6 vertices:
and size 1:  0
and size 2:  0
and size 3:  0
and size 4:  0
and size 5:  0
and size 6:  4
total:  4
Number of structures without these C_11 homomorphic images
and size 5:  6
and size 6:  88


In [2]:
# Explanation: This cell runs the exact flag algebra SDP for the edge density upper bound using the balanced bipartite 0001/0111 construction as the rounding guide.

trip_constr = FGT.blowup_construction(
    target_size,
    ["X0", "X1"],
    edges=[[0, 0, 0, 1], [0, 1, 1, 1]],
    symbolic=True
)
eval_constr = trip_constr.subs([1/2, 1/2])

FGT.optimize(
    edge,
    target_size,
    exact=True,
    file="FourGraph_C11_6vtx",
    construction=eval_constr,
    denom=1024*15*3*5,
    slack_threshold=2e-6
)


Base flags generated, their number is 88
The relevant ftypes are constructed, their number is 3
Block sizes before symmetric/asymmetric change is applied: [2, 16, 16]


Done with mult table for Ftype on 4 points with edges=(0123): : 3it [00:01,  1.78it/s]


Tables finished
Constraints finished
Adjusting table with kernels from construction
Running SDP after kernel correction. Used block sizes are [1, 3, 9, 4, 8, -88, -2]
CSDP 6.2.0
Iter:  0 Ap: 0.00e+00 Pobj:  0.0000000e+00 Ad: 0.00e+00 Dobj:  0.0000000e+00 
Iter:  1 Ap: 1.00e+00 Pobj: -1.7360459e+01 Ad: 7.59e-01 Dobj:  7.9347291e-01 
Iter:  2 Ap: 1.00e+00 Pobj: -1.7965987e+01 Ad: 9.50e-01 Dobj: -3.1848623e-01 
Iter:  3 Ap: 1.00e+00 Pobj: -1.7138293e+01 Ad: 9.30e-01 Dobj: -3.7711522e-01 
Iter:  4 Ap: 1.00e+00 Pobj: -8.6083571e+00 Ad: 6.99e-01 Dobj: -3.8060593e-01 
Iter:  5 Ap: 4.83e-01 Pobj: -6.5797768e+00 Ad: 8.27e-01 Dobj: -3.8012369e-01 
Iter:  6 Ap: 1.00e+00 Pobj: -2.2311273e+00 Ad: 7.82e-01 Dobj: -3.8299105e-01 
Iter:  7 Ap: 1.00e+00 Pobj: -8.3063477e-01 Ad: 7.89e-01 Dobj: -3.9476243e-01 
Iter:  8 Ap: 1.00e+00 Pobj: -6.3946321e-01 Ad: 7.58e-01 Dobj: -4.3234098e-01 
Iter:  9 Ap: 6.12e-01 Pobj: -6.0220937e-01 Ad: 6.71e-01 Dobj: -4.5297468e-01 
Iter: 10 Ap: 1.00e+00 Pobj: -5.6228157e-01

 33%|████████████████████████▎                                                | 1/3 [00:00<00:00, 25.28it/s]


Rounded X matrix ((5, Ftype on 4 points with edges=(), 6), 1) is not semidefinite: -0.14835321699619397
Rounding based on construction was unsuccessful
Rounding X matrices


100%|████████████████████████████████████████████████████████████████████████| 5/5 [00:00<00:00, 594.52it/s]


Calculating resulting bound


100%|█████████████████████████████████████████████████████████████████████████| 3/3 [00:00<00:00, 30.80it/s]

The rounded result is -363463/691200
Final rounded bound is 363463/691200


363463/691200